# meeting_extracted_sem_stage_a — 方向 C Stage A：業務語意判類（規則 ＋ LLM）

- 用途：規格 §8.3 / `_prompt_weekly_extract.md` v1.1 Stage A 的 Databricks 版。讀 `v_meeting_items_sem_cand`（中立層有效條目 ＋ SQL 先驗），把每條判成 `semantic` / `intel` / `event`，寫鏡頭表 `g_meeting_items_sem_bu`（一列一個 `item_key`）。PM 之後直接在表上填 `verified` / `review_note`；Stage B 另做。
- 判法（D30）：`rule_basis = default`（約 89%）**不呼叫 LLM**，直接寫 event；`definition` / `term_hint` / `intel_tag` 交 LLM 判 category / term / route_hint / as_of / extract_note；同文重複（`dup_key` 非空）不呼叫 LLM，複製最早那條的判定。quote 逐字抄 view，不改寫。
- 輸入 View：`v_meeting_items_sem_cand`（003）；輸出表：`g_meeting_items_sem_bu`（003）。MERGE on `item_key`，**不覆蓋 `verified` / `review_note` / `sem_id` / `dup_of`**；中立層已沒有的 key 標 `active = false`，不刪。
- 參數（widgets）：`catalog`、`schema`、`unit`、`week`（空 = 所有未判；`2026-W42` = 只判該週）、`stage_a_version`、`model`、`max_items`（費用閘門，只限制 LLM 候選）、`batch_size`、`concurrency`、`reclassify`、`prompt_cache`、`dry_run`。
- 排程：每週一次，接在 W1 三支之後；第一次等於 backfill（目前全部條目在 2026-W42 / W43）。
- 費用：每批 `batch_size` 條一次呼叫；第一輪約 1,650 條 LLM 候選（含 D30 加的 price / cost / allocation）≈ 85 次呼叫，估 < USD 5。
- 負責人 / 更新日期：（填）/ 2026-10-19

| cell | 內容 |
|---|---|
| [c01] params | widgets |
| [c02] imports | 常數：類別、route_hint 規則、讀哪些欄 |
| [c03] pure_shared | norm_space / normalize_text / sha1_hex（與 W1 三支相同） |
| [c04] prompt_stage_a | system prompt（§8.3 兩題判準 ＋ 投影代號表）、user prompt 組裝 |
| [c05] pure_postprocess | 解析 / 驗證 LLM 輸出、route_hint / as_of 規則 |
| [c06] llm_client | Azure Foundry 接線（同 task 3） |
| [c07] spark_transforms | 待判列選取、規則列、LLM 列合併、同文重複複製、停用 |
| [c10] load_source | 讀 view、切批 |
| [c11] run_classify | 呼叫 LLM |
| [c12] gates | 關卡 ＋ log ＋ dry_run 樣本 |
| [c20] write_lens | MERGE |
| [c21] maintain_lens | 停用消失的 key、刷新 item_id |
| [c30] check_target | 驗證 |


In [ ]:
# [c01] params
# 方向 C Stage A：讀 v_meeting_items_sem_cand 未判列 → 規則列直接寫、LLM 候選分批呼叫 → MERGE g_meeting_items_sem_{unit}。
# 由 job parameters 覆蓋；widget 建立過後改預設值不會生效，要重設先 dbutils.widgets.removeAll()。
import re as _re

dbutils.widgets.text("catalog", "micenter")
dbutils.widgets.text("schema", "mi3_datahub_prod")
dbutils.widgets.text("unit", "bu")                   # 讀寫哪一組（v_meeting_items_sem_cand 目前只有 bu；表名 g_meeting_items_sem_{unit}）
dbutils.widgets.text("week", "")                     # 空 = 所有未判列；"2026-W42" = 只判該週（view 的 week 欄）
dbutils.widgets.text("stage_a_version", "0.1")       # 要等於 [c04] STAGE_A_VERSION_IN_CELL；改 prompt / 規則就升
dbutils.widgets.text("model", "claude-sonnet-5")     # Azure Foundry 上的模型名（同 task 3）
dbutils.widgets.text("max_items", "40")              # 費用閘門：本次最多送幾條給 LLM（只限制 LLM 候選；規則列與同文重複不受限）；0 = 不限
dbutils.widgets.text("batch_size", "20")             # 一次呼叫帶幾條
dbutils.widgets.text("concurrency", "4")
dbutils.widgets.dropdown("reclassify", "false", ["true", "false"])   # true：stage_a_version 不同的已判列也重判（verified / review_note 不動）
dbutils.widgets.dropdown("prompt_cache", "true", ["true", "false"])
dbutils.widgets.dropdown("dry_run", "true", ["true", "false"])      # true：照樣呼叫 LLM（受 max_items 限制）、印關卡與樣本，不寫表

cfg = {
    "catalog": dbutils.widgets.get("catalog").strip(),
    "schema": dbutils.widgets.get("schema").strip(),
    "unit": dbutils.widgets.get("unit").strip(),
    "week": dbutils.widgets.get("week").strip(),
    "stage_a_version": dbutils.widgets.get("stage_a_version").strip(),
    "model": dbutils.widgets.get("model").strip(),
    "max_items": int(dbutils.widgets.get("max_items")),
    "batch_size": int(dbutils.widgets.get("batch_size")),
    "concurrency": max(1, int(dbutils.widgets.get("concurrency"))),
    "reclassify": dbutils.widgets.get("reclassify") == "true",
    "prompt_cache": dbutils.widgets.get("prompt_cache") == "true",
    "dry_run": dbutils.widgets.get("dry_run") == "true",
}
assert cfg["catalog"] and cfg["schema"] and cfg["unit"], "catalog / schema / unit 不可為空"
assert cfg["stage_a_version"] and cfg["model"], "stage_a_version / model 不可為空"
assert not cfg["week"] or _re.fullmatch(r"\d{4}-W\d{2}", cfg["week"]), "week 格式 YYYY-Www，或留空"
assert cfg["max_items"] >= 0 and 1 <= cfg["batch_size"] <= 40 and cfg["concurrency"] <= 16, "max_items ≥ 0；batch_size 1–40；concurrency ≤ 16"
print(cfg)

In [ ]:
# [c02] imports
# [c02]～[c05]、[c07] 是純 Python / 純 DataFrame 函式（不碰 spark session / dbutils / 網路），tests/ 直接載入；
# [c06] 是 LLM 接線。常數集中在這裡；判類規則照規格 §8.3 與 _prompt_weekly_extract.md v1.1。
import hashlib
import json
import re
import time
import unicodedata
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

CATEGORIES = ("semantic", "intel", "event")
APPS = ("TV", "NB", "MNT", "ALL", "OTHER")
# D30：哪些先驗交 LLM、哪些直接寫。rule_basis 值來自 v_meeting_items_sem_cand（003 [c05]）。
LLM_RULE_BASIS = ("definition", "term_hint", "intel_tag")
RULE_CATEGORY = {"default": "event"}          # 不交 LLM 的先驗 → 直接寫這個 category
RULE_MODEL = "rule"                           # 純規則寫入的 model 欄值（003 spec）
# route_hint（SemanticLayerPlan §1.3 投影代號）：semantic 只接受這些前綴，其餘改成預設；intel 由程式組；event NULL。
ROUTE_PREFIXES_SEMANTIC = ("genie:", "de:", "skill:", "pa:", "brief:", "mv:")
ROUTE_DEFAULT_SEMANTIC = "pa:semantics"
ROUTE_INTEL_FMT = "background_intel:{app}"
AS_OF_RE = re.compile(r"^\d{4}-(?:0[1-9]|1[0-2]|Q[1-4])$")   # yyyy-MM 或 yyyy-Qn
MAX_TERM_CHARS = 60
MAX_NOTE_CHARS = 400
MAX_ITEMS_PER_CALL = 40                        # 與 [c01] batch_size 上限一致；LLM 回超過視為整批失敗
EMAIL_ANY_RE = re.compile(r"[A-Za-z0-9._%+\-]+@[A-Za-z0-9\-]+(?:\.[A-Za-z0-9\-]+)+")
# LLM 費用估算（§10.1：Sonnet 5 標價，美元 / 百萬 token）。只印 log。
PRICE_PER_MTOK = {"input": 2.0, "output": 10.0, "cache_write": 2.5, "cache_read": 0.2}

# 從候選 view 讀的欄（送 driver 組 prompt 的只有 LLM 候選；規則列與同文重複全在 Spark 做）
CAND_COLS = ["item_id", "item_key", "pk_seq", "segment_no", "sub_no", "unit", "week", "category_rule", "rule_basis",
             "app", "customer", "customer_raw", "statement_type", "topic_tag_primary", "topic_tags_str", "free_tags_str",
             "terms_str", "summary", "quote", "date", "event_month", "source", "tags", "speaker_side", "content_type",
             "weight_hint", "sensitivity", "confidence", "already_classified", "sem_active", "sem_stage_a_version",
             "dup_key"]
# 判類結果欄（三種來源：規則 / LLM / 同文複製 都要有）；sa_confidence 寫進表時改名 confidence（view 的 confidence 是 L1 自評）
CLS_COLS = ["category", "term", "route_hint", "as_of", "extract_note", "sa_confidence", "model"]
CLS_SCHEMA = T.StructType([
    T.StructField("item_key", T.StringType(), False),
    T.StructField("category", T.StringType(), True),
    T.StructField("term", T.StringType(), True),
    T.StructField("route_hint", T.StringType(), True),
    T.StructField("as_of", T.StringType(), True),
    T.StructField("extract_note", T.StringType(), True),
    T.StructField("sa_confidence", T.DoubleType(), True),
    T.StructField("model", T.StringType(), True),
])
# 鏡頭表裡 Stage A 不碰的欄（PM / Stage B 的）；MERGE 的 UPDATE / INSERT 都不含
PROTECTED_COLS = ("verified", "review_note", "sem_id", "dup_of")

In [ ]:
# [c03] pure_shared
# 與 w1_t1_mail / w1_t1_clean / w1_t2_extract [c03] 相同的純函式。複製而不是 import：Databricks 只能貼 cell。
def norm_space(s: str | None) -> str:
    """所有空白（含 U+00A0、換行）壓成一格並去頭尾。"""
    return re.sub(r"\s+", " ", (s or "").replace("\xa0", " ")).strip()


def normalize_text(s: str | None) -> str:
    """去重 / 穩定鍵共用的正規化：NFKC（全半形統一）、去標點、去空白、去控制字元、小寫。"""
    t = unicodedata.normalize("NFKC", s or "")
    return "".join(ch for ch in t if unicodedata.category(ch)[0] not in "PZC").lower()


def sha1_hex(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8")).hexdigest()

In [ ]:
# [c04] prompt_stage_a
# Stage A 判類 prompt：_prompt_weekly_extract.md v1.1 Stage A 的兩題判準 ＋ SemanticLayerPlan §1.3 投影代號，改成「對已萃出的原子條目判類」
# （輸入不是原始素材，quote 由程式逐字抄 view，LLM 不輸出 quote / summary）。prompt 不帶姓名 / email（條目已 token 化）。
# 改這個 cell 就升 [c01] stage_a_version。
STAGE_A_VERSION_IN_CELL = "0.1"  # [c11] 會比對 cfg["stage_a_version"]，不一致就停

ROUTE_TABLE = """【semantic 的 route_hint（投影代號，只能選一個）】
skill:multi_genie  查詢口徑的紅旗（分母、單位、哪個指標才是對的）
pa:semantics       領域術語 / 驗證項目 / 代碼的定義（最常用）
de:主控            跨 skill 的通則（財年起算、客戶稱呼對應）
genie:客戶會議紀錄  會議紀錄本身的欄位語意
brief:過濾卡       daily brief 要過濾 / 要留的規則
mv:<view>          可做成 Metric View 的 KPI 口徑（寫成 mv:kpi名）"""

FEW_SHOT = """【示例】（取自其他條目，只示範判法）
條目：fact / quality，先驗 semantic/definition，summary「說明 System matching 是用來驗證前端系統版給 Panel 訊號/Power 電性是否合乎 AUO PS 規格，無法測試出內部 EC OP/Buck/LDO 相關問題」
→ {"n": 1, "category": "semantic", "term": "System matching 驗證範圍", "route_hint": "pa:semantics", "as_of": null, "extract_note": "跨機種通用的驗證項目口徑，一年後仍成立", "confidence": 0.9}
條目：opinion / capacity，先驗 intel/intel_tag，summary「推測 INX 目前 TV 產能集中於 32、43、50 吋，因成本上升市場價格無法同步調漲導致需求不佳，可能轉移產能至 23.8 吋 Monitor 面板」，日期 2026-08-28
→ {"n": 2, "category": "intel", "term": "INX 產能可能轉 23.8 吋 MNT", "route_hint": null, "as_of": "2026-Q3", "extract_note": "對手產能動向，未來幾個月解讀 MNT 供需都會用到", "confidence": 0.85}
條目：fact / spec，先驗 semantic/definition，summary「M238HAN05.6D0 FD spec 同 DVT stage，right spec for 黑PI = JND2.2」
→ {"n": 3, "category": "event", "term": "M238HAN05.6D0 FD spec 設定", "route_hint": null, "as_of": null, "extract_note": "單一機種的 spec 設定，不是跨專案口徑；先驗 definition 推翻", "confidence": 0.8}
條目：fact / supply_chain，先驗 intel/intel_tag，summary「受影響料號 46.S0966.625(F3) 變更為 46.S0966.576(F4)」
→ {"n": 4, "category": "event", "term": "料號 46.S0966 F3→F4 變更", "route_hint": null, "as_of": null, "extract_note": "單次 EC 料號變更，沒有持續的供應限制；先驗 intel 推翻", "confidence": 0.9}
條目：fact / supply_chain，先驗 intel/intel_tag，summary「Common Cell 項目：AUO 針對 IC 以同一型號名稱導入雙來源，但 VD 認為 SMIC 不可、nextchip 與 chipone 可以」，日期 2026-06-22
→ {"n": 5, "category": "intel", "term": "VD 對 IC 第二來源的接受度", "route_hint": null, "as_of": "2026-06", "extract_note": "客戶對供應商的限制會持續影響後續 IC 選用", "confidence": 0.85}"""

SYSTEM_PROMPT_SA = f"""你是 AUO DSBG「MI 語意 / 情報判類」的執行者。輸入是已從客戶會議紀錄萃出的原子條目（每條有 summary、quote、日期、來源、程式先驗），人名都已換成 <Pnn> token。你的工作：對每條判 category，並給 term / route_hint / as_of / extract_note / confidence。你只產候選，不定案（verified 是 PM 的欄位）。不要嘗試還原任何人的姓名。

【分類判準——每條先過這兩題】
Q1 這句話一年後還成立嗎？
   成立 → semantic（話怎麼講、數字怎麼算：詞彙、口徑、換算規則、驗證項目的定義、代碼的意義、來源權威性）
Q2 不成立，但未來幾個月每次解讀數字都會用到嗎？
   會   → intel（判讀框架：客戶策略與態度、供需狀態、產品線動向、持續的產能 / 料況 / 供應商限制）
   不會 → event（單點事實：某筆訂單、某週報價、某次驗證結果、某個料號變更、某機種的 spec 設定）

【常見誤判】
- 「定義」不等於 semantic：只適用單一機種 / 單一專案的規則（某機種 FD spec 為 JND2.2、某次驗證只需確認良率）是 event；跨機種跨專案都適用的口徑（System matching 驗什麼、分貨 code 怎麼定、FY 怎麼算、BP 的口徑）才是 semantic。
- EC 會議的料號變更 / EOL / LTB 通知是 event，除非它描述的是會持續幾個月的供應限制（某 IC 只有兩家來源、客戶不接受某供應商、某晶圓廠停產影響整個系列）→ intel。
- 數字本身是 event（本月 allocation 20,000 片）；數字背後的解讀框架（客戶因缺料採內部分配、對手因成本轉產能）才是 intel。
- 「已完成」不等於 event：完成後持續有效的狀態（TV PCB 已全數移到中國廠區、某產線已關、某客戶已改單一供應商）之後解讀數字都會用到 → intel；一次性的交付 / 確認 / 金額（claim 金額確認、本次需求數字更新）才是 event。
- 政策宣導（反托拉斯、資安）是 event，不是 semantic。
- 程式先驗（rule）只是提示，可以推翻；推翻時在 extract_note 寫為什麼。

【欄位】
- term：≤ 20 字短語，讓人掃一眼知道這條講什麼（semantic 寫詞彙 / 口徑名；intel 寫主題；event 寫事件）。
- route_hint：semantic → 下表的投影代號之一；intel 與 event → null（intel 的去向由程式依 app 填）。
- as_of：intel 必填，"yyyy-MM" 或 "yyyy-Qn"，用條目的日期推；semantic / event → null。
- extract_note：一句判類理由；與先驗不同時說明；內外訊號矛盾也寫這裡。不寫給 PM 的話。不要引用 n 編號（n 只在本批有效），要指同批另一條就寫它的 term。
- confidence：0–1 自評。

{ROUTE_TABLE}

【輸出】只輸出一個 JSON 物件，不要 markdown 圍欄、不要說明文字：
{{"items": [{{"n": 1, "category": "semantic|intel|event", "term": "", "route_hint": null, "as_of": null, "extract_note": "", "confidence": 0.0}}]}}
每個輸入條目都要有一個對應的 n，不可漏、不可多；n 照輸入。

{FEW_SHOT}"""


def build_user_prompt(items: list[dict]) -> str:
    """一批條目（view 列 dict，已 token 化）→ user prompt。n 從 1 起照順序；[c05] 用 n 對回 item_key。"""
    blocks = []
    for n, it in enumerate(items, start=1):
        head = (f"### n={n} | 先驗 {it.get('category_rule')}/{it.get('rule_basis')} | {it.get('statement_type')} / "
                f"{it.get('topic_tag_primary')} | app {it.get('app')} | 客戶 {it.get('customer') or it.get('customer_raw') or '無'} | "
                f"日期 {it.get('date')} | 段類型 {it.get('content_type')}")
        lines = [head, f"來源: {norm_space(it.get('source'))}", f"summary: {norm_space(it.get('summary'))}",
                 f"quote: {norm_space(it.get('quote'))}"]
        if it.get("terms_str"):
            lines.append(f"命中術語: {it['terms_str']}")
        if it.get("free_tags_str"):
            lines.append(f"關鍵字: {it['free_tags_str']}")
        blocks.append("\n".join(lines))
    return f"## 本批 {len(items)} 條，請逐條判類\n\n" + "\n\n".join(blocks)


def retry_user_prompt(error: str) -> str:
    """JSON / schema 失敗後的第二輪 user 訊息（接在上一輪 assistant 輸出之後）。"""
    return f"上一輪輸出無法使用：{error}。請重新只輸出一個合法的 JSON 物件（同樣的 schema，items 的 n 要與輸入一一對應），不要任何說明文字或 markdown 圍欄。"

In [ ]:
# [c05] pure_postprocess
# LLM 回覆 → 判類列。程式說了算的部分：category 只能三選一；intel 的 route_hint 由 app 組、as_of 不合格式就用 event_month；
# semantic 的 route_hint 不在投影代號表就改預設；event 的 route_hint / as_of 一律 NULL；term 空就拿 summary 開頭。
# 漏掉的 n 不產列（下次再判）；多出來的 n 忽略。term / extract_note 裡的 email 樣式換成 <email>（S3 硬規則）。
def parse_llm_json(text: str | None) -> tuple[dict | None, str | None]:
    """回覆文字 → (dict, None) 或 (None, 錯誤訊息)。容忍 markdown 圍欄與前後說明文字（取第一個 { 到最後一個 }）。"""
    if not text or not text.strip():
        return None, "empty response"
    s = text.strip()
    s = re.sub(r"^```(?:json)?\s*", "", s)
    s = re.sub(r"\s*```$", "", s)
    i, j = s.find("{"), s.rfind("}")
    if i < 0 or j <= i:
        return None, "no JSON object found"
    try:
        obj = json.loads(s[i:j + 1])
    except json.JSONDecodeError as e:
        return None, f"JSONDecodeError: {e.msg} at pos {e.pos}"
    if not isinstance(obj, dict) or not isinstance(obj.get("items"), list):
        return None, "top-level must be an object with items list"
    return obj, None


def _str_or_none(v, cap: int) -> str | None:
    if v is None or isinstance(v, (list, dict)):
        return None
    s = norm_space(str(v))
    return s[:cap] if s else None


def _mask_email(s: str | None) -> str | None:
    return EMAIL_ANY_RE.sub("<email>", s) if s else s


def as_of_from_month(event_month: str | None) -> str | None:
    """view 的 event_month（yyyy-MM）→ as_of；缺就 None。"""
    m = (event_month or "").strip()
    return m if AS_OF_RE.match(m) else None


def route_for(category: str, app: str | None, llm_route) -> tuple[str | None, bool]:
    """回傳 (route_hint, 是否被程式改過)。semantic：LLM 給的要有投影代號前綴，否則預設；intel：程式組；event：NULL。"""
    if category == "intel":
        return ROUTE_INTEL_FMT.format(app=app if app in APPS else "ALL"), False
    if category == "semantic":
        r = _str_or_none(llm_route, 80)
        if r and r.startswith(ROUTE_PREFIXES_SEMANTIC):
            return r, False
        return ROUTE_DEFAULT_SEMANTIC, True
    return None, False


def classify_one(out: dict, it: dict) -> tuple[dict | None, dict]:
    """單條 LLM 輸出 ＋ 對應的 view 列 → (判類列 dict 或 None, 統計)。None = 這條作廢（category 不合法）。"""
    st: dict = {}
    cat = (out.get("category") or "").strip().lower() if isinstance(out.get("category"), str) else ""
    if cat not in CATEGORIES:
        st["bad_category"] = 1
        return None, st
    term = _mask_email(_str_or_none(out.get("term"), MAX_TERM_CHARS))
    if not term:
        term = _str_or_none(it.get("summary"), 40)
        st["term_filled"] = 1
    route, fixed = route_for(cat, it.get("app"), out.get("route_hint"))
    if fixed:
        st["route_fixed"] = 1
    as_of = None
    if cat == "intel":
        v = _str_or_none(out.get("as_of"), 10)
        as_of = v if v and AS_OF_RE.match(v) else as_of_from_month(it.get("event_month"))
        if not (v and AS_OF_RE.match(v)):
            st["as_of_fixed"] = 1
    note = _mask_email(_str_or_none(out.get("extract_note"), MAX_NOTE_CHARS))
    conf = out.get("confidence")
    try:
        conf = min(1.0, max(0.0, float(conf))) if conf is not None and not isinstance(conf, bool) else None
    except (TypeError, ValueError):
        conf = None
    if cat != it.get("category_rule"):
        st["disagree"] = 1
    st[cat] = 1
    return {"item_key": it["item_key"], "category": cat, "term": term, "route_hint": route, "as_of": as_of,
            "extract_note": note, "sa_confidence": conf, "model": None}, st


def build_cls_rows(resp: dict | None, error: str | None, items: list[dict], model: str) -> tuple[list[dict], dict]:
    """一批的 LLM 結果 → 判類列（CLS_SCHEMA 欄）＋ 統計。resp None = 整批失敗（error 記統計，不產列）。"""
    st: dict = {"sent": len(items)}
    if resp is None:
        st["batch_failed"] = 1
        st["failed_items"] = len(items)
        st["errors"] = [error]
        return [], st
    outs = resp.get("items") or []
    if len(outs) > MAX_ITEMS_PER_CALL:
        st["batch_failed"] = 1
        st["failed_items"] = len(items)
        st["errors"] = [f"too many items: {len(outs)}"]
        return [], st
    by_n: dict[int, dict] = {}
    for o in outs:
        if isinstance(o, dict) and isinstance(o.get("n"), int) and not isinstance(o.get("n"), bool):
            if o["n"] in by_n:
                st["dup_n"] = st.get("dup_n", 0) + 1
            by_n.setdefault(o["n"], o)
    rows: list[dict] = []
    for n, it in enumerate(items, start=1):
        o = by_n.get(n)
        if o is None:
            st["missing"] = st.get("missing", 0) + 1
            continue
        row, s1 = classify_one(o, it)
        for k, v in s1.items():
            st[k] = st.get(k, 0) + v
        if row:
            row["model"] = model
            rows.append(row)
    st["extra"] = len([n for n in by_n if n < 1 or n > len(items)])
    st["classified"] = len(rows)
    return rows, st


def merge_stats(total: dict, part: dict) -> dict:
    for k, v in part.items():
        if isinstance(v, list):
            total[k] = total.get(k, []) + v
        else:
            total[k] = total.get(k, 0) + v
    return total

In [ ]:
# [c06] llm_client
# LLM 接線：Azure Foundry 的 Anthropic Messages 端點，與 task 3 [c06] 同一組 secret / URL。用 requests（DBR 內建），不 %pip install。
# thinking 關閉、不送 temperature（Claude 5 系列已移除）、system 開 prompt caching（system prompt 固定，整批共用）。
# 重試：HTTP 408 / 429 / 5xx / 連線錯誤指數退避 3 次；4xx 其他直接回 error 不重試。JSON / schema 失敗的重試在 [c11]。
import requests

FOUNDRY_URL = "https://micenter.services.ai.azure.com/anthropic/v1/messages"
FOUNDRY_API_KEY = dbutils.secrets.get(scope="llm-api-keys", key="ms-foundry")
SA_MAX_TOKENS = 8000      # 一批 ≤ 40 條、每條 ≤ 150 token 的 JSON，8k 夠；截斷視為整批失敗（[c11]）
HTTP_TIMEOUT_S = 180


def call_llm(messages: list[dict], system_prompt: str, model: str, cache: bool) -> dict:
    """回傳 {"text", "stop_reason", "usage": {input, output, cache_read, cache_write}, "latency_s", "error"}；
    error 非 None 代表端點層失敗（已重試），text 為 None。"""
    system = [{"type": "text", "text": system_prompt}]
    if cache:
        system[0]["cache_control"] = {"type": "ephemeral"}
    body = {"model": model, "max_tokens": SA_MAX_TOKENS, "system": system,
            "messages": messages, "thinking": {"type": "disabled"}}
    headers = {"x-api-key": FOUNDRY_API_KEY, "anthropic-version": "2023-06-01", "content-type": "application/json"}
    t0 = time.time()
    err = None
    for attempt in range(3):
        try:
            resp = requests.post(FOUNDRY_URL, headers=headers, json=body, timeout=HTTP_TIMEOUT_S)
            if resp.status_code in (408, 429) or resp.status_code >= 500:
                err = f"HTTP {resp.status_code}: {resp.text[:200]}"
            elif resp.status_code >= 400:
                return {"text": None, "stop_reason": None, "usage": {}, "latency_s": round(time.time() - t0, 2),
                        "error": f"HTTP {resp.status_code}: {resp.text[:300]}"}
            else:
                data = resp.json()
                text = "\n".join(b.get("text", "") for b in data.get("content", []) if b.get("type") == "text")
                u = data.get("usage") or {}
                usage = {"input": u.get("input_tokens", 0) or 0, "output": u.get("output_tokens", 0) or 0,
                         "cache_read": u.get("cache_read_input_tokens", 0) or 0,
                         "cache_write": u.get("cache_creation_input_tokens", 0) or 0}
                return {"text": text, "stop_reason": data.get("stop_reason"), "usage": usage,
                        "latency_s": round(time.time() - t0, 2), "error": None}
        except (requests.RequestException, ValueError) as e:
            err = f"{type(e).__name__}: {str(e)[:200]}"
        time.sleep(5 * (2 ** attempt))
    return {"text": None, "stop_reason": None, "usage": {}, "latency_s": round(time.time() - t0, 2), "error": err}


print("llm client ready:", FOUNDRY_URL.split("/anthropic")[0], "| model:", cfg["model"], "| cache:", cfg["prompt_cache"])

In [ ]:
# [c07] spark_transforms
# 純 DataFrame 函式：待判列選取、三路拆分（規則 / LLM / 同文重複）、規則列判類、同文複製、組成鏡頭表列。不 cache（serverless）。
# 從 view 複製進鏡頭表的欄（view 欄名 = 表欄名；view 的 confidence 是 L1 自評，不複製，鏡頭表 confidence 是 Stage A 自評）
COPY_COLS = ["item_id", "item_key", "pk_seq", "segment_no", "sub_no", "unit", "week", "category_rule", "rule_basis",
             "app", "customer", "summary", "quote", "date", "source", "tags", "dup_key", "statement_type",
             "topic_tag_primary", "content_type", "sensitivity"]


def select_pending(df_cand: DataFrame, stage_a_version: str, reclassify: bool, week: str) -> DataFrame:
    """待判 = 鏡頭表沒有的 key、或鏡頭表標 active = false 但中立層又出現的 key；reclassify 再加上 stage_a_version 不同的。
    week 非空只取該週。**一個 item_key 只留一列**（item_id 最小）：中立層同一段同一句 quote 萃出兩條時 item_key 相同
    （item_key = sha1(pk‖seg‖quote)），不去重會讓 join 變平方、MERGE 多源列報錯（2026-10-19 dry_run 踩到）。"""
    cond = ~F.col("already_classified") | (F.col("sem_active") == F.lit(False))
    if reclassify:
        cond = cond | (F.coalesce(F.col("sem_stage_a_version"), F.lit("")) != F.lit(stage_a_version))
    df = df_cand.where(cond)
    if week:
        df = df.where(F.col("week") == F.lit(week))
    w = Window.partitionBy("item_key").orderBy("item_id")
    return df.withColumn("_rn", F.row_number().over(w)).where(F.col("_rn") == 1).drop("_rn")


def split_pending(df: DataFrame) -> tuple[DataFrame, DataFrame, DataFrame]:
    """→ (規則列, LLM 候選, 同文重複)。同文重複優先（不管先驗），其餘看 rule_basis 是否在 LLM_RULE_BASIS。"""
    is_dup = F.col("dup_key").isNotNull()
    is_llm = F.col("rule_basis").isin(*LLM_RULE_BASIS)
    return df.where(~is_dup & ~is_llm), df.where(~is_dup & is_llm), df.where(is_dup)


def _cls_select(df: DataFrame, category, term, route_hint, as_of, extract_note, sa_confidence, model) -> DataFrame:
    return df.select(
        "item_key", category.alias("category"), term.alias("term"), route_hint.alias("route_hint"), as_of.alias("as_of"),
        extract_note.alias("extract_note"), sa_confidence.alias("sa_confidence"), model.alias("model"),
    )


def rule_cls(df_rule: DataFrame) -> DataFrame:
    """規則列的判類欄（item_key ＋ CLS_COLS）：category 照 RULE_CATEGORY，表裡沒有的 rule_basis 保守給 event；
    term / route_hint / as_of / confidence NULL；extract_note = rule:<basis>；model = rule。"""
    cat = F.lit("event")
    for k, v in RULE_CATEGORY.items():
        cat = F.when(F.col("rule_basis") == F.lit(k), F.lit(v)).otherwise(cat)
    null_s, null_d = F.lit(None).cast("string"), F.lit(None).cast("double")
    return _cls_select(df_rule, cat, null_s, null_s, null_s, F.concat(F.lit("rule:"), F.col("rule_basis")), null_d,
                       F.lit(RULE_MODEL))


def lens_cls_existing(df_lens: DataFrame) -> DataFrame:
    """鏡頭表既有 active 列的判類欄，給同文複製當來源。"""
    return df_lens.where(F.col("active")).select(
        "item_key", "category", "term", "route_hint", "as_of", "extract_note",
        F.col("confidence").alias("sa_confidence"), "model",
    )


def dup_cls(df_dup: DataFrame, df_primary_cls: DataFrame) -> DataFrame:
    """同文重複：複製最早那條（dup_key）的判定；最早那條還沒判（不在 df_primary_cls）→ 這條這次不產列，下次再來。
    intel 的 as_of 用自己的 event_month（較晚出現 = 較新的有效基準）、route_hint 依自己的 app 重組；extract_note = dup of <key>。"""
    p = df_primary_cls.select(F.col("item_key").alias("dup_key"), *[F.col(c).alias(f"_p_{c}") for c in CLS_COLS])
    j = df_dup.join(p, "dup_key", "inner")
    own_month = F.when(F.col("event_month").rlike(r"^\d{4}-(0[1-9]|1[0-2])$"), F.col("event_month"))
    is_intel = F.col("_p_category") == F.lit("intel")
    as_of = F.when(is_intel, F.coalesce(own_month, F.col("_p_as_of")))
    route = F.when(is_intel, F.concat(F.lit("background_intel:"), F.coalesce(F.col("app"), F.lit("ALL")))).otherwise(
        F.col("_p_route_hint"))
    return _cls_select(j, F.col("_p_category"), F.col("_p_term"), route, as_of,
                       F.concat(F.lit("dup of "), F.col("dup_key")), F.col("_p_sa_confidence"), F.col("_p_model"))


def to_lens_rows(df_rows: DataFrame, df_cls: DataFrame, stage_a_version: str) -> DataFrame:
    """view 列 ⋈ 判類欄（inner；沒判到的列不出）→ 鏡頭表欄（PROTECTED_COLS 除外）。"""
    now = F.current_timestamp()
    return (
        df_rows.select(*COPY_COLS).join(df_cls, "item_key", "inner")
        .select(*COPY_COLS, "category", "term", "route_hint", "as_of", "extract_note",
                F.col("sa_confidence").alias("confidence"), F.lit(True).alias("active"),
                F.lit(stage_a_version).alias("stage_a_version"), "model", now.alias("classified_at"),
                now.alias("updated_at"))
    )

In [ ]:
# [c10] load_source
# 讀候選 view（003）→ 待判列 → 三路：規則列（全寫）、LLM 候選（受 max_items 限制，collect 到 driver）、同文重複（複製）。
def fq(name: str) -> str:
    return f"{cfg['catalog']}.{cfg['schema']}.{name}"


fq_cand, fq_lens = fq("v_meeting_items_sem_cand"), fq(f"g_meeting_items_sem_{cfg['unit']}")
for t in (fq_cand, fq_lens):
    assert spark.catalog.tableExists(t), f"{t} 不存在：先跑 migration 003"
df_cand = spark.table(fq_cand).where(F.col("unit") == F.lit(cfg["unit"]))
missing = sorted(set(CAND_COLS) - set(df_cand.columns))
assert not missing, f"候選 view 缺欄 {missing}：重跑 003（dup_key / sem_active / sem_stage_a_version 是 2026-10-19 加的）"
lens_tbl = spark.table(fq_lens)
assert "dup_key" in lens_tbl.columns, (
    f"{fq_lens} 缺 dup_key（2026-10-19 加進 003 spec）：表還是空的就 DROP TABLE 後重跑 003；有資料則 ALTER TABLE ADD COLUMNS (dup_key STRING)")

df_pending = select_pending(df_cand, cfg["stage_a_version"], cfg["reclassify"], cfg["week"])
df_rule, df_llm, df_dup = split_pending(df_pending)
n_rule, n_llm_all, n_dup_all = df_rule.count(), df_llm.count(), df_dup.count()
# dry_run 隨機抽（不然 max_items 永遠是同幾場會議的前幾條）；正式跑照週、日期順序
df_llm_sel = df_llm.orderBy(F.rand(7)) if cfg["dry_run"] else df_llm.orderBy("week", "date", "item_key")
if cfg["max_items"] > 0:
    df_llm_sel = df_llm_sel.limit(cfg["max_items"])
items = [r.asDict() for r in df_llm_sel.select(*CAND_COLS).collect()]
scope = (f"week {cfg['week'] or 'ALL'} | reclassify {cfg['reclassify']} | pending {n_rule + n_llm_all + n_dup_all}"
         f" = rule {n_rule} + llm {n_llm_all} + dup {n_dup_all} | llm sent {len(items)}"
         + (f"（max_items {cfg['max_items']}）" if cfg["max_items"] else ""))
print(scope)

In [ ]:
# [c11] run_classify
# LLM 候選分批（batch_size 條一次）在 driver 呼叫（ThreadPool）。這是刻意的例外：LLM 呼叫是 I/O 等待，筆數受 max_items 限制。
# 每批：user prompt → 呼叫 → 解析；JSON / schema 失敗把上一輪輸出當 assistant turn、帶錯誤訊息再試 1 次；
# 輸出被 max_tokens 截斷則整批作廢（下次再判）。端點失敗的批不產列。
# 然後在 Spark 組本批：規則列 ∪ LLM 列 ∪ 同文複製 → 鏡頭表欄；最後 collect 釘住（view 在寫表後會變，lazy 重算會變 0 列；task 2 [c11] 教訓）。
assert STAGE_A_VERSION_IN_CELL == cfg["stage_a_version"], (
    f"[c04] 版本 {STAGE_A_VERSION_IN_CELL} ≠ widget stage_a_version {cfg['stage_a_version']}")

bs = cfg["batch_size"]
batches = [items[i:i + bs] for i in range(0, len(items), bs)]
usage_tot = {"input": 0, "output": 0, "cache_read": 0, "cache_write": 0, "calls": 0, "retries": 0}
latencies: list[float] = []


def _account(r: dict) -> None:
    usage_tot["calls"] += 1
    for k in ("input", "output", "cache_read", "cache_write"):
        usage_tot[k] += (r.get("usage") or {}).get(k, 0)
    latencies.append(r.get("latency_s") or 0.0)


def process_batch(batch: list[dict]) -> dict:
    user_prompt = build_user_prompt(batch)
    messages = [{"role": "user", "content": user_prompt}]
    r1 = call_llm(messages, SYSTEM_PROMPT_SA, cfg["model"], cfg["prompt_cache"])
    _account(r1)
    if r1["error"]:
        rows, st = build_cls_rows(None, r1["error"], batch, cfg["model"])
        return {"rows": rows, "stats": st, "error": r1["error"]}
    truncated = r1["stop_reason"] == "max_tokens"
    obj, perr = (None, "output truncated at max_tokens") if truncated else parse_llm_json(r1["text"])
    if obj is None and not truncated:
        usage_tot["retries"] += 1
        messages += [{"role": "assistant", "content": (r1["text"] or "").strip() or "(empty)"},
                     {"role": "user", "content": retry_user_prompt(perr)}]
        r2 = call_llm(messages, SYSTEM_PROMPT_SA, cfg["model"], cfg["prompt_cache"])
        _account(r2)
        if r2["error"]:
            rows, st = build_cls_rows(None, r2["error"], batch, cfg["model"])
            return {"rows": rows, "stats": st, "error": r2["error"]}
        obj, perr = (None, "truncated (retry)") if r2["stop_reason"] == "max_tokens" else parse_llm_json(r2["text"])
    rows, st = build_cls_rows(obj, perr, batch, cfg["model"])
    return {"rows": rows, "stats": st, "error": None}


PROGRESS_EVERY = 10
results: list[dict] = []
if batches:
    t_start, n_done = time.time(), 0
    results = [None] * len(batches)
    with ThreadPoolExecutor(max_workers=cfg["concurrency"]) as ex:
        futs = {ex.submit(process_batch, b): i for i, b in enumerate(batches)}
        for fut in as_completed(futs):
            results[futs[fut]] = fut.result()
            n_done += 1
            if n_done % PROGRESS_EVERY == 0 or n_done == len(batches):
                print(f"progress {n_done}/{len(batches)} batches | {time.time() - t_start:.0f}s | calls {usage_tot['calls']} | "
                      f"retries {usage_tot['retries']} | endpoint errors {sum(1 for r in results if r and r['error'])}", flush=True)
cls_rows = [row for r in results for row in r["rows"]]
df_llm_cls = spark.createDataFrame([tuple(r[f] for f in CLS_SCHEMA.fieldNames()) for r in cls_rows], CLS_SCHEMA)

df_primary_cls = rule_cls(df_rule).unionByName(df_llm_cls)
df_dup_cls = dup_cls(df_dup, df_primary_cls.unionByName(lens_cls_existing(lens_tbl)))
lens_cols = [c for c in lens_tbl.schema.fieldNames() if c not in PROTECTED_COLS]
df_batch = to_lens_rows(df_pending, df_primary_cls.unionByName(df_dup_cls), cfg["stage_a_version"]).select(*lens_cols)
batch_rows = df_batch.collect()
df_batch = spark.createDataFrame(batch_rows, df_batch.schema)
df_batch.createOrReplaceTempView("_sem_stage_a_batch")
n_batch = len(batch_rows)
batch_keys = [r["item_key"] for r in batch_rows]
n_dup_written = sum(1 for r in batch_rows if (r["extract_note"] or "").startswith("dup of "))
print(f"llm classified {len(cls_rows)}/{len(items)} | batch rows {n_batch} = rule {n_rule} + llm {len(cls_rows)} + dup {n_dup_written}"
      f"（dup 等最早那條：{n_dup_all - n_dup_written}）| calls {usage_tot['calls']} | retries {usage_tot['retries']}")

In [ ]:
# [c12] gates
# 關卡：端點全掛 → 不寫半批；LLM 漏判（missing）> 30% 停；category 不合法 > 5% WARN。只印：先驗 vs LLM 的不一致率
# （rule_disagree；太高代表先驗或 prompt 有一邊要調）、route_hint / as_of 被程式改掉的數、category 分布、token 用量與費用。
def pct(n, d) -> float | None:
    return round(n / d, 3) if d else None


tot: dict = {}
for r in results:
    merge_stats(tot, r["stats"])
n_sent, n_batches = len(items), len(batches)
n_batch_failed, n_classified = tot.get("batch_failed", 0), tot.get("classified", 0)
n_missing, n_bad = tot.get("missing", 0), tot.get("bad_category", 0)
cost = (usage_tot["input"] * PRICE_PER_MTOK["input"] + usage_tot["output"] * PRICE_PER_MTOK["output"]
        + usage_tot["cache_read"] * PRICE_PER_MTOK["cache_read"] + usage_tot["cache_write"] * PRICE_PER_MTOK["cache_write"]) / 1e6
llm_rows = [r for r in batch_rows if r["model"] != RULE_MODEL and not (r["extract_note"] or "").startswith("dup of ")]
log = {
    "scope": scope, "stage_a_version": cfg["stage_a_version"], "model": cfg["model"],
    "batch_rows": n_batch, "rule_rows": n_rule, "dup_rows": n_dup_written, "dup_waiting": n_dup_all - n_dup_written,
    "llm": {"sent": n_sent, "batches": n_batches, "batches_failed": n_batch_failed, "classified": n_classified,
            "missing": n_missing, "bad_category": n_bad, "missing_rate": pct(n_missing, n_sent),
            "disagree_rate": pct(tot.get("disagree", 0), n_classified),
            "route_fixed": tot.get("route_fixed", 0), "as_of_fixed": tot.get("as_of_fixed", 0),
            "term_filled": tot.get("term_filled", 0),
            "category_dist": {c: tot.get(c, 0) for c in CATEGORIES},
            **usage_tot, "latency_p50_s": (sorted(latencies)[len(latencies) // 2] if latencies else None),
            "latency_max_s": (max(latencies) if latencies else None), "est_cost_usd": round(cost, 3)},
    "batch_category_dist": dict(Counter(r["category"] for r in batch_rows)),
    "llm_rule_vs_category": dict(Counter(f"{r['category_rule']}->{r['category']}" for r in llm_rows)),
}
print(json.dumps(log, ensure_ascii=False, default=str, indent=1))
if tot.get("errors"):
    print("失敗的批（最多 5）：", tot["errors"][:5])
if n_sent:
    assert n_batch_failed < n_batches, "LLM 端點整體不可用：全部批失敗，不寫半批"
    assert n_batch_failed / n_batches <= 0.5, f"批失敗 {n_batch_failed}/{n_batches} 超過一半：先查端點再跑（失敗的下次會再撿）"
    if (log["llm"]["missing_rate"] or 0) > 0.10:
        print(f"WARN LLM 漏判 {log['llm']['missing_rate']:.1%} > 10%：看 [c04] 的 n 對應規則或縮小 batch_size")
    assert (log["llm"]["missing_rate"] or 0) <= 0.30, "LLM 漏判 > 30%：先查 prompt / batch_size 再跑"
    if pct(n_bad, n_sent) and pct(n_bad, n_sent) > 0.05:
        print(f"WARN category 不合法 {n_bad}/{n_sent} > 5%")
    if (log["llm"]["disagree_rate"] or 0) > 0.6:
        print(f"WARN LLM 與先驗不一致 {log['llm']['disagree_rate']:.0%}：先驗規則（003 [c02]）或 prompt 有一邊該調")
if n_batch == 0:
    print("本批 0 列：沒有待判條目，後面 cell 不寫表。")
if cfg["dry_run"] and llm_rows:
    print("\ndry_run = true：不寫表。LLM 判的樣本（先驗不同的排前面；已 token 化，無人名）：")
    (df_batch.where((F.col("model") != RULE_MODEL) & ~F.col("extract_note").startswith("dup of "))
     .select("item_key", "rule_basis", "category_rule", "category", "term", "route_hint", "as_of", "confidence",
             F.substring("extract_note", 1, 120).alias("note_head"), F.substring("summary", 1, 120).alias("summary_head"))
     .orderBy((F.col("category") == F.col("category_rule")).asc(), "item_key").limit(20).show(truncate=100, vertical=True))

In [ ]:
# [c20] write_lens
# MERGE on item_key：UPDATE / INSERT 都只碰 Stage A 的欄（PROTECTED_COLS = verified / review_note / sem_id / dup_of 不在清單裡，
# 既有值原封不動；新列這四欄 NULL）。同文重複與重判（reclassify）走同一條路。
n_verified_before = None
if not cfg["dry_run"] and n_batch > 0:
    n_verified_before = spark.table(fq_lens).where(F.col("verified").isNotNull()).count()
    upd = ", ".join(f"t.{c} = s.{c}" for c in lens_cols if c != "item_key")
    ins_cols, ins_vals = ", ".join(lens_cols), ", ".join(f"s.{c}" for c in lens_cols)
    spark.sql(f"""
        MERGE INTO {fq_lens} t
        USING _sem_stage_a_batch s
        ON t.item_key = s.item_key
        WHEN MATCHED THEN UPDATE SET {upd}
        WHEN NOT MATCHED THEN INSERT ({ins_cols}) VALUES ({ins_vals})
    """)
    print(f"merged {n_batch} rows →", fq_lens, f"| verified rows before: {n_verified_before}")
else:
    print("skip write（dry_run 或本批 0 列）")

In [ ]:
# [c21] maintain_lens
# 每次跑都做（不限本批）：(1) 鏡頭表 active 的 key 已不在候選 view（中立層重萃後 quote 變、或條目被 superseded）→ active = false，不刪；
# (2) 同 key 但中立層 item_id 變了 → 刷新 item_id。都是小表 MERGE。候選 view 0 列時跳過 (1)（中立層重建中不能把整張表停用）。
# 候選 view 同一 item_key 可能多列（同段同句萃兩條；[c07] 去重留 item_id 最小）→ 這裡的來源也要先 GROUP BY item_key 取 min(item_id)，
# 否則 MERGE 報 DELTA_MULTIPLE_SOURCE_ROW_MATCHING_TARGET_ROW（2026-10-19 全量首跑踩到）。
if not cfg["dry_run"]:
    cand_sql = (f"SELECT item_key, min(item_id) AS item_id FROM {fq_cand} WHERE unit = '{cfg['unit']}' "
                f"GROUP BY item_key")
    n_cand = df_cand.count()
    if n_cand == 0:
        print("候選 view 0 列：跳過停用檢查")
        n_deact = 0
    else:
        n_deact = spark.sql(f"SELECT count(*) AS n FROM {fq_lens} s LEFT ANTI JOIN ({cand_sql}) v ON v.item_key = s.item_key "
                            f"WHERE s.active = true").first()["n"]
        if n_deact:
            spark.sql(f"""
                MERGE INTO {fq_lens} t
                USING (SELECT s.item_key FROM {fq_lens} s LEFT ANTI JOIN ({cand_sql}) v ON v.item_key = s.item_key
                       WHERE s.active = true) m
                ON t.item_key = m.item_key
                WHEN MATCHED THEN UPDATE SET t.active = false, t.updated_at = current_timestamp()
            """)
    n_refresh = spark.sql(f"SELECT count(*) AS n FROM ({cand_sql}) v JOIN {fq_lens} s ON s.item_key = v.item_key "
                          f"WHERE s.item_id <> v.item_id").first()["n"]
    if n_refresh:
        spark.sql(f"""
            MERGE INTO {fq_lens} t
            USING (SELECT v.item_key, v.item_id FROM ({cand_sql}) v JOIN {fq_lens} s ON s.item_key = v.item_key
                   WHERE s.item_id <> v.item_id) m
            ON t.item_key = m.item_key
            WHEN MATCHED THEN UPDATE SET t.item_id = m.item_id, t.updated_at = current_timestamp()
        """)
    print(f"maintain: candidates {n_cand} | deactivated {n_deact} | item_id refreshed {n_refresh}")
else:
    print("skip maintain（dry_run）")

In [ ]:
# [c22] sync_neutral
# 每次跑都做（不限本批）：鏡頭表複製自中立層的 customer / sensitivity / tags，若中立層事後被程式改了（task 3 [c22] 規則補客戶、
# [c23] 重算敏感度；都不重萃、item_key 不變），這裡同步回鏡頭表，否則 digest / Genie 看鏡頭表還是舊值（D33）。
# 只動這三欄；判類欄與 PM 欄不碰。來源同 [c21]：候選 view 同 item_key 取 item_id 最小那列。dry_run 只印筆數。
_src = f"""SELECT item_key, customer, sensitivity, tags FROM (
             SELECT item_key, customer, sensitivity, tags, row_number() OVER (PARTITION BY item_key ORDER BY item_id) AS rn
             FROM {fq_cand} WHERE unit = '{cfg['unit']}') WHERE rn = 1"""
_stale = (f"SELECT s.item_key, v.customer, v.sensitivity, v.tags FROM {fq_lens} s JOIN ({_src}) v ON v.item_key = s.item_key "
          f"WHERE s.active AND (NOT (s.customer <=> v.customer) OR NOT (s.sensitivity <=> v.sensitivity) OR NOT (s.tags <=> v.tags))")
n_stale = spark.sql(f"SELECT count(*) AS n FROM ({_stale})").first()["n"]
if n_stale and not cfg["dry_run"]:
    spark.sql(f"""
        MERGE INTO {fq_lens} t USING ({_stale}) m ON t.item_key = m.item_key
        WHEN MATCHED THEN UPDATE SET t.customer = m.customer, t.sensitivity = m.sensitivity, t.tags = m.tags,
                                     t.updated_at = current_timestamp()
    """)
print(f"sync neutral cols：customer / sensitivity / tags 與中立層不同 {n_stale} 列" + ("（dry_run 不寫）" if cfg["dry_run"] else "，已同步"))


In [ ]:
# [c30] check_target
# 寫入後驗證（用 [c11] 釘住的 batch_keys，不重算 view）：本批 key 都在表、版本正確；item_key 無重複；
# verified 列數沒變少（MERGE 沒碰 PM 欄）；intel 都有 as_of、semantic / intel 都有 route_hint；category 合法；
# term / extract_note 無 email 樣式（S3）。最後印表的 active 分布。
if not cfg["dry_run"] and n_batch > 0:
    lens = spark.table(fq_lens)
    keys_df = spark.createDataFrame([(k,) for k in batch_keys], "item_key string")
    got = lens.join(keys_df, "item_key", "left_semi")
    email_rx = r"[A-Za-z0-9._%+\-]+@[A-Za-z0-9\-]+\.[A-Za-z]{2,}"
    chk = {
        "batch_rows_in_table": got.count(), "expected": n_batch,
        "batch_version_ok": got.where(F.col("stage_a_version") == F.lit(cfg["stage_a_version"])).count(),
        "dup_item_key": lens.groupBy("item_key").count().where("count > 1").count(),
        "verified_before": n_verified_before,
        "verified_after": lens.where(F.col("verified").isNotNull()).count(),
        "intel_missing_as_of": got.where((F.col("category") == "intel") & F.col("as_of").isNull()).count(),
        "sem_intel_missing_route": got.where(F.col("category").isin("semantic", "intel") & F.col("route_hint").isNull()).count(),
        "bad_category": got.where(~F.col("category").isin(*CATEGORIES)).count(),
        "email_in_batch": got.where(F.col("term").rlike(email_rx) | F.col("extract_note").rlike(email_rx)).count(),
        "table_total": lens.count(),
        "table_active": lens.where(F.col("active")).count(),
        "active_category_dist": {r["category"]: r["count"]
                                 for r in lens.where(F.col("active")).groupBy("category").count().collect()},
    }
    print(chk)
    assert chk["batch_rows_in_table"] == chk["expected"] == chk["batch_version_ok"], "本批 key 沒全寫進表或版本不對"
    assert chk["dup_item_key"] == 0, "item_key 重複（MERGE 鍵壞了）"
    assert chk["verified_after"] >= chk["verified_before"], "verified 列數變少：MERGE 碰到 PM 欄"
    assert chk["intel_missing_as_of"] == 0 and chk["sem_intel_missing_route"] == 0, "intel 缺 as_of 或 semantic / intel 缺 route_hint"
    assert chk["bad_category"] == 0 and chk["email_in_batch"] == 0, "category 不合法或殘留 email"